# 🔬 NanoDes: RepAF-Denoise Net (Interactive Google Colab Notebook)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kilugit/nanodes/blob/main/NanoDes_Colab.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-kilugit%2Fnanodes-181717.svg?logo=github)](https://github.com/kilugit/nanodes)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-EE4C2C.svg?logo=pytorch&logoColor=white)](https://pytorch.org/)
[![ONNX](https://img.shields.io/badge/ONNX-Runtime-005CED.svg?logo=onnx&logoColor=white)](https://onnxruntime.ai/)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](https://github.com/kilugit/nanodes)

---

### **Overview**
**NanoDes** is a lightweight, ultra-fast image denoising and artifact-removal network based on **RepAF-Denoise Net** (Re-parameterized Asymmetric Feature Denoise Net) with 2D Haar Wavelet Transforms and Multi-Branch Structural Re-parameterization.

This interactive notebook completely replaces the desktop GUI (`gui.py` and `export_gui.py`), enabling seamless execution on **Google Colab** (with free GPU acceleration: T4, A100, V100) or local Jupyter environments.

#### **Key Features in this Notebook**:
1. **Interactive Denoising & Inference (Replaces `gui.py` Tab 1)**:
   - Load sample images or upload your own high-resolution photos.
   - Simulate realistic degradation (progressive JPEG compression, downsampling, and sensor/Gaussian noise).
   - High-throughput tiled inference with Hann window blending (supports 4K/8K images without VRAM limits).
   - Visual inspection: side-by-side comparisons, difference heatmaps, and zoom-in ROI analysis.
2. **ONNX Export & Quantization Suite (Replaces `export_gui.py` & `export.py`)**:
   - Re-parameterization numerical equivalence verification ($< 10^{-5}$ error).
   - FP32 and FP16 ONNX export with dynamic axes `[batch_size, height, width]`.
   - INT8 QDQ static quantization with representative sample calibration.
   - Symmetric per-channel INT8 PyTorch quantization.
   - Cross-backend performance benchmarks (PyTorch vs ONNX vs INT8) & model download package.
3. **Dataset Generation Pipeline (Replaces `gui.py` Tab 3)**:
   - Extract balanced crops from high-resolution source images with JPEG/noise augmentation.
4. **Two-Stage Progressive Training (Replaces `gui.py` Tab 2)**:
   - **Stage 1**: Coarse training with progressive patch scaling ($128 \to 256$) with `Stage1Loss` (Charbonnier + Sobel gradient loss).
   - **Stage 2**: Fine-tuning on $512 \times 512$ patches with `PSNRLoss`.

In [ ]:
#@title ⚙️ 1. Setup Environment & Check Hardware
#@markdown Run this cell to configure directory structures, install required dependencies, and inspect hardware.

import os
import sys
import glob
import time
import shutil
from typing import Optional, Tuple, List, Dict

# Detect if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ

Git_Repo_URL = "https://github.com/kilugit/nanodes.git" #@param {type:"string"}
Auto_Clone_If_Missing = True #@param {type:"boolean"}
Mount_Google_Drive = False #@param {type:"boolean"}

if IN_COLAB:
    print("🚀 Detected Google Colab environment.")
    if Mount_Google_Drive:
        from google.colab import drive
        drive.mount("/content/drive")
        
    # Check repository location
    if not os.path.exists("models.py"):
        if os.path.exists("nanodes/models.py"):
            os.chdir("nanodes")
            print(f"Switched working directory to: {os.getcwd()}")
        elif os.path.exists("NanoDes/models.py"):
            os.chdir("NanoDes")
            print(f"Switched working directory to: {os.getcwd()}")
        elif Git_Repo_URL and Auto_Clone_If_Missing:
            print(f"Cloning repository from {Git_Repo_URL}...")
            !git clone {Git_Repo_URL}
            if os.path.exists("nanodes/models.py"):
                os.chdir("nanodes")
            elif os.path.exists("NanoDes/models.py"):
                os.chdir("NanoDes")
            print(f"Switched working directory to: {os.getcwd()}")
                
    # Install required packages
    !pip install -q onnx onnxruntime onnxruntime-gpu pillow matplotlib
else:
    print("💻 Running in standard Python/Jupyter environment.")

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Verify and create standard project directories
REQUIRED_DIRS = [
    "samples",
    "outputs",
    "pytorch models",
    "onnx models",
    "samples/processed/clean",
    "samples/processed/noisy",
    "samples/processed/train/clean",
    "samples/processed/train/noisy",
    "samples/processed/val/clean",
    "samples/processed/val/noisy",
]
for d in REQUIRED_DIRS:
    os.makedirs(d, exist_ok=True)

# Hardware Diagnostic
print("\n" + "=" * 50)
print("HARDWARE STATUS:")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  [✓] GPU Accelerated: {gpu_name}")
    print(f"  [✓] Available VRAM:  {vram_gb:.2f} GB")
    print(f"  [✓] PyTorch CUDA:    {torch.version.cuda}")
else:
    print("  [!] GPU not detected. Using CPU execution backend.")
print(f"  [✓] Working Directory: {os.getcwd()}")
print("=" * 50)

# Check if repository modules exist in path
missing_modules = []
for mod in ["models", "inference", "export", "dataset", "losses", "metrics", "process_samples"]:
    if not os.path.exists(f"{mod}.py"):
        missing_modules.append(mod)

if missing_modules:
    print(f"⚠️ Warning: The following Python modules are missing in {os.getcwd()}: {missing_modules}")
    print("Make sure you are in the repository folder containing models.py and inference.py.")
else:
    print("✓ All NanoDes core modules verified.")

---
## 🖼️ Section 2: Interactive Denoising & Inference (Replaces GUI Tab 1)
Load images, inject realistic camera sensor noise and JPEG compression artifacts, and execute denoise inference using either PyTorch checkpoints or ONNX models.

In [ ]:
#@title 🖼️ 2.1 Select or Upload Input Image
#@markdown Choose whether to select an existing sample image from `samples/` or upload your own image.

Image_Source = "Select from samples" #@param ["Select from samples", "Upload from local computer", "Use direct file path"]
Sample_Image = "samples/image-544.png" #@param {type:"string"}
Custom_File_Path = "" #@param {type:"string"}

active_image_path = None
active_clean_image = None

if Image_Source == "Select from samples":
    # Discover available sample images
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp")
    found_samples = []
    for ext in exts:
        found_samples.extend(glob.glob(os.path.join("samples", ext)))
    found_samples = [f for f in found_samples if not f.startswith("samples" + os.sep + "processed")]
    
    if os.path.exists(Sample_Image):
        active_image_path = Sample_Image
    elif found_samples:
        active_image_path = sorted(found_samples)[0]
        print(f"'{Sample_Image}' not found. Defaulting to first discovered sample: {active_image_path}")
    else:
        # Create a synthetic high-res sample if none exist
        active_image_path = "samples/sample_synthetic.png"
        synth_arr = np.zeros((768, 1024, 3), dtype=np.uint8)
        for y in range(768):
            for x in range(1024):
                synth_arr[y, x, 0] = int(255 * (x / 1024))
                synth_arr[y, x, 1] = int(255 * (y / 768))
                synth_arr[y, x, 2] = int(255 * ((x + y) / (1024 + 768)))
        Image.fromarray(synth_arr).save(active_image_path)
        print(f"Created synthetic test image: {active_image_path}")

elif Image_Source == "Upload from local computer":
    if IN_COLAB:
        from google.colab import files
        print("Please choose an image file to upload:")
        uploaded = files.upload()
        if uploaded:
            uploaded_filename = list(uploaded.keys())[0]
            active_image_path = os.path.join("samples", uploaded_filename)
            shutil.move(uploaded_filename, active_image_path)
            print(f"Uploaded and saved to: {active_image_path}")
        else:
            raise ValueError("No file uploaded!")
    else:
        print("Upload widget is designed for Google Colab. For local Jupyter, enter the path below.")
        active_image_path = Custom_File_Path

elif Image_Source == "Use direct file path":
    active_image_path = Custom_File_Path
    if not os.path.exists(active_image_path):
        raise FileNotFoundError(f"File not found: {active_image_path}")

# Load image
with Image.open(active_image_path) as raw_img:
    active_clean_image = raw_img.convert("RGB")

active_noisy_image = active_clean_image.copy()
w, h = active_clean_image.size
print(f"✓ Successfully loaded: '{active_image_path}'")
print(f"  Resolution: {w} x {h} pixels | Megapixels: {(w * h) / 1e6:.2f} MP")

# Preview image
plt.figure(figsize=(8, 5))
plt.imshow(active_clean_image)
plt.title(f"Loaded Source Image: {os.path.basename(active_image_path)} ({w}x{h})")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#@title 🧪 2.2 Optional: Simulate Degradation (JPEG Artifacts & Noise)
#@markdown Like the desktop GUI, you can inject camera sensor noise and JPEG compression to test the denoiser on fresh artifacts.

Inject_Degradation = True #@param {type:"boolean"}
JPEG_Quality = 35 #@param {type:"slider", min:10, max:95, step:5}
Downsample_Scale = 0.85 #@param {type:"slider", min:0.5, max:1.0, step:0.05}
Sensor_Noise_Sigma = 0.02 #@param {type:"slider", min:0.0, max:0.10, step:0.005}
Noise_Type = "sensor" #@param ["sensor", "gaussian", "poisson", "none"]

from process_samples import (
    add_realistic_noise,
    apply_downsample_upsample,
    apply_jpeg_compression,
)
from metrics import calculate_psnr, calculate_ssim

if Inject_Degradation and active_clean_image is not None:
    degraded = active_clean_image.copy()
    if Downsample_Scale < 1.0:
        degraded = apply_downsample_upsample(degraded, scale=Downsample_Scale)
    if JPEG_Quality < 100:
        degraded = apply_jpeg_compression(degraded, quality=JPEG_Quality)
    if Sensor_Noise_Sigma > 0.0 and Noise_Type != "none":
        degraded = add_realistic_noise(degraded, noise_type=Noise_Type, sigma=Sensor_Noise_Sigma)
    
    active_noisy_image = degraded
    print(f"✓ Injected Degradation: JPEG Q={JPEG_Quality}, Downsample={Downsample_Scale}x, {Noise_Type.capitalize()} Noise σ={Sensor_Noise_Sigma}")
    
    # Calculate degraded quality vs original clean
    clean_t = torch.from_numpy(np.array(active_clean_image)).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    noisy_t = torch.from_numpy(np.array(active_noisy_image)).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    initial_psnr = calculate_psnr(noisy_t, clean_t)
    initial_ssim = calculate_ssim(noisy_t, clean_t)
    print(f"  Degraded Input PSNR: {initial_psnr:.2f} dB | SSIM: {initial_ssim:.4f}")

    # Side-by-side preview
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(active_clean_image)
    axes[0].set_title("Clean Reference")
    axes[0].axis("off")
    
    axes[1].imshow(active_noisy_image)
    axes[1].set_title(f"Degraded Input (PSNR: {initial_psnr:.2f} dB | Q={JPEG_Quality})")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    active_noisy_image = active_clean_image.copy()
    print("Using pristine loaded image as input (no synthetic degradation applied).")

In [ ]:
#@title 🚀 2.3 Run Denoising Inference
#@markdown Select your model checkpoint (.pth or .onnx) and execution options.

# Discover available models in pytorch models/ and onnx models/
available_models = []
for folder in ["pytorch models", "onnx models"]:
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.endswith((".pth", ".onnx")):
                available_models.append(os.path.join(folder, f))

default_model = available_models[0] if available_models else "pytorch models/best_model.pth"

Model_Path = "pytorch models/best_model.pth" #@param {type:"string"}
Device_Backend = "Auto" #@param ["Auto", "GPU", "CPU"]
Tiled_Mode = True #@param {type:"boolean"}
Tile_Size = 1024 #@param [512, 1024, 2048] {type:"raw"}
Tile_Overlap = 64 #@param {type:"integer"}

from inference import run_denoise

# Check model path
if not os.path.exists(Model_Path):
    # Try finding candidate
    candidates = [
        "pytorch models/best_model.pth",
        "pytorch models/repaf_denoise_net_deployed.pth",
        "onnx models/repaf_denoise_net.onnx",
        "onnx models/repaf_denoise_net_int8.onnx",
    ]
    for c in candidates:
        if os.path.exists(c):
            Model_Path = c
            print(f"Selected model not found. Automatically using found candidate: {Model_Path}")
            break

if not os.path.exists(Model_Path):
    raise FileNotFoundError(f"Model file '{Model_Path}' does not exist! Please train a model or run Section 3 ONNX export.")

# Resolve device
dev_str = Device_Backend.lower()
if dev_str == "auto":
    dev_str = "gpu" if torch.cuda.is_available() else "cpu"

print(f"Executing Denoise Inference:")
print(f"  Model:   {Model_Path}")
print(f"  Backend: {dev_str.upper()}")
print(f"  Tiled:   {Tiled_Mode} (Tile Size: {Tile_Size}x{Tile_Size}, Overlap: {Tile_Overlap})")
w, h = active_noisy_image.size

denoised_image, elapsed_ms = run_denoise(
    image=active_noisy_image,
    model_path=Model_Path,
    tiled=Tiled_Mode,
    tile_size=Tile_Size,
    tile_overlap=Tile_Overlap,
    device=dev_str,
)

fps = 1000.0 / elapsed_ms if elapsed_ms > 0 else 0.0
mpix_per_sec = ((w * h) / 1e6) / (elapsed_ms / 1000.0) if elapsed_ms > 0 else 0.0

print(f"\n✓ Denoising Complete!")
print(f"  Image Size:   {w} x {h} ({w*h/1e6:.2f} Megapixels)")
print(f"  Latency:      {elapsed_ms:.1f} ms")
print(f"  Throughput:   {fps:.2f} FPS ({mpix_per_sec:.2f} MPix/sec)")

In [ ]:
#@title 🔍 2.4 Visualize Comparison & High-Resolution Quality Inspection
#@markdown Compares Input vs Denoised Output, renders an amplified difference map, and zooms into a region of interest.

Zoom_Center_X_Percent = 50 #@param {type:"slider", min:10, max:90, step:5}
Zoom_Center_Y_Percent = 50 #@param {type:"slider", min:10, max:90, step:5}
Zoom_Crop_Size = 256 #@param [128, 256, 384, 512] {type:"raw"}

w, h = denoised_image.size
noisy_np = np.array(active_noisy_image).astype(np.float32) / 255.0
denoised_np = np.array(denoised_image).astype(np.float32) / 255.0

# Calculate removed residual (amplified for visibility)
diff_np = np.abs(denoised_np - noisy_np)
diff_amplified = np.clip(diff_np * 5.0, 0.0, 1.0)

# Compute metrics if clean reference is available
metrics_text = ""
if active_clean_image is not None:
    clean_t = torch.from_numpy(np.array(active_clean_image)).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    denoised_t = torch.from_numpy(denoised_np).permute(2, 0, 1).unsqueeze(0).float()
    psnr_out = calculate_psnr(denoised_t, clean_t)
    ssim_out = calculate_ssim(denoised_t, clean_t)
    metrics_text = f"PSNR: {psnr_out:.2f} dB | SSIM: {ssim_out:.4f}"
    print(f"Restoration Quality: {metrics_text}")

# 1. Full Image Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(active_noisy_image)
axes[0].set_title(f"Input Image ({w}x{h})")
axes[0].axis("off")

axes[1].imshow(denoised_image)
axes[1].set_title(f"Denoised Output ({metrics_text})")
axes[1].axis("off")

axes[2].imshow(diff_amplified)
axes[2].set_title("Removed Noise Residual (5x Amplified)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

# 2. Zoomed ROI Comparison
cx = int(w * (Zoom_Center_X_Percent / 100.0))
cy = int(h * (Zoom_Center_Y_Percent / 100.0))
half = Zoom_Crop_Size // 2
x0 = max(0, min(w - Zoom_Crop_Size, cx - half))
y0 = max(0, min(h - Zoom_Crop_Size, cy - half))
x1 = x0 + Zoom_Crop_Size
y1 = y0 + Zoom_Crop_Size

crop_noisy = active_noisy_image.crop((x0, y0, x1, y1))
crop_denoised = denoised_image.crop((x0, y0, x1, y1))
crop_clean = active_clean_image.crop((x0, y0, x1, y1)) if active_clean_image else None

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
if crop_clean:
    axes[0].imshow(crop_clean)
    axes[0].set_title("Original Clean Reference (Crop)")
else:
    axes[0].imshow(crop_noisy)
    axes[0].set_title("Input (Crop)")
axes[0].axis("off")

axes[1].imshow(crop_noisy)
axes[1].set_title(f"Degraded Input (Crop [{x0}:{x1}, {y0}:{y1}])")
axes[1].axis("off")

axes[2].imshow(crop_denoised)
axes[2].set_title("NanoDes Denoised Output (Crop)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#@title 💾 2.5 Save & Download Output Image
#@markdown Save the resulting image to disk and optionally trigger an automatic browser download.

Output_Filename = "denoised_result.png" #@param {type:"string"}
Download_To_Computer = True #@param {type:"boolean"}

os.makedirs("outputs", exist_ok=True)
output_path = os.path.join("outputs", Output_Filename)
denoised_image.save(output_path, format="PNG")
print(f"✓ Saved denoised image to: {output_path} ({os.path.getsize(output_path) / 1024:.1f} KB)")

if Download_To_Computer:
    if IN_COLAB:
        from google.colab import files
        print("Initiating browser download...")
        files.download(output_path)
    else:
        print(f"File saved locally at: {os.path.abspath(output_path)}")

---
## 📦 Section 3: ONNX Export & Quantization Suite (Replaces `export_gui.py`)
This section provides complete structural re-parameterization, ONNX export (FP32, FP16), and QDQ static INT8 quantization capabilities.

### **The Structural Re-parameterization Principle:**
During **training**, the network uses a multi-branch architecture:
$$\text{Output} = \text{Conv}_{3\times 3}(x) + \text{Conv}_{1\times 1}(x) + x$$
This multi-branch topology enriches the loss landscape and accelerates convergence.

During **deployment**, because 2D convolution and identity mapping are linear operators, these three parallel paths can be analytically collapsed into **one single $3\times 3$ convolution**:
$$W_{\text{deploy}} = W_{3\times 3} + \text{pad}(W_{1\times 1}) + \text{pad}(I)$$
$$B_{\text{deploy}} = B_{3\times 3} + B_{1\times 1} + B_{\text{identity}}$$
This eliminates memory traffic, reduces FLOPs, and enables optimal hardware execution in ONNX Runtime and TensorRT!

In [ ]:
#@title 🧪 3.1 Structural Re-parameterization Equivalence Test
#@markdown Mathematically verifies that the fused deployment model produces identical outputs to the multi-branch training model.

Test_Device = "cpu" #@param ["cpu", "cuda"]
Numerical_Tolerance = 1e-5 #@param {type:"number"}

from models import RepAFDenoiseNet, test_reparameterization_equivalence

dev = Test_Device if (Test_Device == "cpu" or torch.cuda.is_available()) else "cpu"
print(f"Running Re-parameterization Analytical Equivalence Test on {dev.upper()}...")

max_error = test_reparameterization_equivalence(device=dev, tol=Numerical_Tolerance)

print(f"\n" + "=" * 55)
print("RE-PARAMETERIZATION VERIFICATION RESULT:")
print(f"  Maximum Absolute Numerical Error: {max_error:.8e}")
print(f"  Error Threshold (Tolerance):     {Numerical_Tolerance:.8e}")
if max_error < Numerical_Tolerance:
    print(f"  Status: PASSED! (Error is negligible, mathematically equivalent)")
else:
    print(f"  Status: FAILED (Exceeded tolerance)")
print("=" * 55)

In [ ]:
#@title 📦 3.2 Export Model to ONNX (FP32 & FP16)
#@markdown Exports your PyTorch checkpoint to ONNX with dynamic input/output axes for batch size and resolution.

Source_Checkpoint = "pytorch models/best_model.pth" #@param {type:"string"}
Export_FP32 = True #@param {type:"boolean"}
Export_FP16 = True #@param {type:"boolean"}
Opset_Version = 17 #@param [14, 15, 16, 17, 18] {type:"raw"}

from export import export_onnx, load_checkpoint_model
import onnx

os.makedirs("onnx models", exist_ok=True)
os.makedirs("pytorch models", exist_ok=True)

# Load PyTorch checkpoint & prepare deployment model
print(f"Loading PyTorch checkpoint: {Source_Checkpoint}")
model = load_checkpoint_model(Source_Checkpoint)

# Save fused deployed PyTorch weights
deployed_pth_path = "pytorch models/repaf_denoise_net_deployed.pth"
torch.save(model.state_dict(), deployed_pth_path)
print(f"✓ Saved fused PyTorch deployment weights to: {deployed_pth_path}")

exported_files = []

# 1. FP32 ONNX Export
if Export_FP32:
    fp32_onnx_path = "onnx models/repaf_denoise_net.onnx"
    print(f"\nExporting FP32 ONNX -> {fp32_onnx_path}...")
    export_onnx(model, save_path=fp32_onnx_path, opset_version=Opset_Version, fp16=False)
    
    # Verify ONNX model
    onnx_m = onnx.load(fp32_onnx_path)
    onnx.checker.check_model(onnx_m)
    sz_mb = os.path.getsize(fp32_onnx_path) / (1024 ** 2)
    print(f"✓ FP32 ONNX export verified! Model size: {sz_mb:.2f} MB")
    exported_files.append(fp32_onnx_path)

# 2. FP16 ONNX Export
if Export_FP16:
    fp16_onnx_path = "onnx models/repaf_denoise_net_fp16.onnx"
    print(f"\nExporting FP16 ONNX -> {fp16_onnx_path}...")
    export_onnx(model, save_path=fp16_onnx_path, opset_version=Opset_Version, fp16=True)
    
    onnx_m16 = onnx.load(fp16_onnx_path)
    onnx.checker.check_model(onnx_m16)
    sz_mb16 = os.path.getsize(fp16_onnx_path) / (1024 ** 2)
    print(f"✓ FP16 ONNX export verified! Model size: {sz_mb16:.2f} MB")
    exported_files.append(fp16_onnx_path)

print(f"\n🎉 Successfully exported {len(exported_files)} ONNX model(s) to 'onnx models/'!")

In [ ]:
#@title ⚡ 3.3 Export to INT8 Quantization (ONNX QDQ & PyTorch INT8)
#@markdown Performs post-training static INT8 quantization (QDQ format) with calibration data from `samples/`.

Export_ONNX_INT8 = True #@param {type:"boolean"}
Export_PyTorch_INT8 = True #@param {type:"boolean"}
Samples_Directory_For_Calibration = "samples" #@param {type:"string"}

from export import export_int8_quantization, export_onnx_int8, load_checkpoint_model
import onnxruntime as ort

model = load_checkpoint_model("pytorch models/best_model.pth")

# 1. ONNX INT8 QDQ Quantization
if Export_ONNX_INT8:
    int8_onnx_path = "onnx models/repaf_denoise_net_int8.onnx"
    print(f"Performing Static QDQ INT8 Quantization using '{Samples_Directory_For_Calibration}'...")
    export_onnx_int8(
        model_or_path=model,
        save_path=int8_onnx_path,
        samples_dir=Samples_Directory_For_Calibration,
        input_shape=(1, 3, 256, 256),
    )
    sz_int8 = os.path.getsize(int8_onnx_path) / (1024 ** 2)
    print(f"✓ ONNX INT8 model exported successfully! Size: {sz_int8:.2f} MB")
    
    # Test session run
    sess = ort.InferenceSession(int8_onnx_path, providers=["CPUExecutionProvider"])
    dummy = np.random.randn(1, 3, 256, 256).astype(np.float32)
    _ = sess.run(None, {"noisy_image": dummy})
    print("✓ ONNX INT8 session execution verified.")

# 2. PyTorch INT8 Quantization
if Export_PyTorch_INT8:
    int8_pth_path = "pytorch models/repaf_denoise_net_int8.pth"
    print(f"\nExporting PyTorch Symmetric Per-Channel INT8 Checkpoint...")
    export_int8_quantization(model, save_path=int8_pth_path)
    sz_pth_int8 = os.path.getsize(int8_pth_path) / (1024 ** 2)
    print(f"✓ PyTorch INT8 checkpoint saved: {int8_pth_path} ({sz_pth_int8:.2f} MB)")

In [ ]:
#@title ⏱️ 3.4 Inference Benchmark & Speed Comparison
#@markdown Benchmarks execution latency across PyTorch (Multi-branch), PyTorch (Deployed), ONNX FP32, and ONNX INT8.

Benchmark_Resolution = 512 #@param [256, 512, 1024] {type:"raw"}
Warmup_Iterations = 3 #@param {type:"integer"}
Benchmark_Iterations = 10 #@param {type:"integer"}

import onnxruntime as ort
from models import RepAFDenoiseNet
from export import load_checkpoint_model

test_shape = (1, 3, Benchmark_Resolution, Benchmark_Resolution)
results = {}

# 1. PyTorch Multi-branch (Training mode)
model_train = RepAFDenoiseNet(c=40).eval()
x_torch = torch.randn(*test_shape)
with torch.inference_mode():
    for _ in range(Warmup_Iterations):
        _ = model_train(x_torch)
    t0 = time.time()
    for _ in range(Benchmark_Iterations):
        _ = model_train(x_torch)
    lat_train = ((time.time() - t0) / Benchmark_Iterations) * 1000.0
results["PyTorch (Multi-branch)"] = lat_train

# 2. PyTorch Deployed (Fused single-conv)
model_deploy = load_checkpoint_model("pytorch models/best_model.pth")
with torch.inference_mode():
    for _ in range(Warmup_Iterations):
        _ = model_deploy(x_torch)
    t0 = time.time()
    for _ in range(Benchmark_Iterations):
        _ = model_deploy(x_torch)
    lat_deploy = ((time.time() - t0) / Benchmark_Iterations) * 1000.0
results["PyTorch (Deploy Fused)"] = lat_deploy

# 3. ONNX FP32
if os.path.exists("onnx models/repaf_denoise_net.onnx"):
    sess_fp32 = ort.InferenceSession("onnx models/repaf_denoise_net.onnx", providers=["CPUExecutionProvider"])
    x_np = np.random.randn(*test_shape).astype(np.float32)
    in_name = sess_fp32.get_inputs()[0].name
    for _ in range(Warmup_Iterations):
        _ = sess_fp32.run(None, {in_name: x_np})
    t0 = time.time()
    for _ in range(Benchmark_Iterations):
        _ = sess_fp32.run(None, {in_name: x_np})
    lat_onnx_fp32 = ((time.time() - t0) / Benchmark_Iterations) * 1000.0
    results["ONNX Runtime (FP32)"] = lat_onnx_fp32

# 4. ONNX INT8
if os.path.exists("onnx models/repaf_denoise_net_int8.onnx"):
    sess_int8 = ort.InferenceSession("onnx models/repaf_denoise_net_int8.onnx", providers=["CPUExecutionProvider"])
    x_np = np.random.randn(*test_shape).astype(np.float32)
    in_name = sess_int8.get_inputs()[0].name
    for _ in range(Warmup_Iterations):
        _ = sess_int8.run(None, {in_name: x_np})
    t0 = time.time()
    for _ in range(Benchmark_Iterations):
        _ = sess_int8.run(None, {in_name: x_np})
    lat_onnx_int8 = ((time.time() - t0) / Benchmark_Iterations) * 1000.0
    results["ONNX Runtime (INT8 QDQ)"] = lat_onnx_int8

# Print Table
print("=" * 60)
print(f"BENCHMARK RESULTS (Input: {Benchmark_Resolution}x{Benchmark_Resolution}, CPU Execution):")
print(f"{'Backend':<28} | {'Latency (ms)':<14} | {'Speedup':<10}")
print("-" * 60)
baseline = results.get("PyTorch (Multi-branch)", 1.0)
for k, v in results.items():
    speedup = baseline / v if v > 0 else 1.0
    print(f"{k:<28} | {v:10.2f} ms   | {speedup:6.2f}x")
print("=" * 60)

# Plot Benchmark Chart
plt.figure(figsize=(9, 4.5))
colors = ["#e57373", "#64b5f6", "#81c784", "#ba68c8"]
bars = plt.bar(list(results.keys()), list(results.values()), color=colors[:len(results)], width=0.55)
plt.title(f"Inference Latency ({Benchmark_Resolution}x{Benchmark_Resolution}) - Lower is Better", fontsize=13, fontweight="bold")
plt.ylabel("Latency (milliseconds)")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + max(results.values())*0.02, f"{yval:.1f} ms", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
#@title 📥 3.5 Package & Download Exported Models
#@markdown Compresses all exported PyTorch checkpoints and ONNX models into a single zip file for deployment.

Archive_Name = "NanoDes_Models_Export.zip" #@param {type:"string"}
Download_Now = True #@param {type:"boolean"}

import zipfile

zip_path = Archive_Name
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in ["pytorch models", "onnx models"]:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for f in files:
                    file_path = os.path.join(root, f)
                    arcname = os.path.relpath(file_path, ".")
                    zipf.write(file_path, arcname)
                    print(f"Added to archive: {arcname}")

sz_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"\n✓ Successfully created archive: '{zip_path}' ({sz_mb:.2f} MB)")

if Download_Now:
    if IN_COLAB:
        from google.colab import files
        print("Starting browser download of exported models package...")
        files.download(zip_path)
    else:
        print(f"Archive saved locally at: {os.path.abspath(zip_path)}")

---
## ✂️ Section 4: Sample Data Generation Pipeline (Replaces GUI Tab 3)
Scans high-resolution images in `samples/` and generates paired clean-noisy training and validation patches with non-aggressive cropping, progressive JPEG compression levels, and realistic sensor noise.

In [ ]:
#@title ✂️ 4.1 Generate Paired Training Dataset
#@markdown Configure sample extraction and degradation parameters.

Crops_Per_Image = 3 #@param {type:"integer"}
Quality_Mode = "Discrete Qualities" #@param ["Discrete Qualities", "Random Quality Range"]
Discrete_Qualities = "30 50 75" #@param {type:"string"}
Random_Versions_Per_Crop = 5 #@param {type:"integer"}
Quality_Min = 20 #@param {type:"integer"}
Quality_Max = 80 #@param {type:"integer"}
Clean_Output_Dir = "samples/processed/clean" #@param {type:"string"}
Noisy_Output_Dir = "samples/processed/noisy" #@param {type:"string"}

import re
from process_samples import process_all_samples

os.makedirs(Clean_Output_Dir, exist_ok=True)
os.makedirs(Noisy_Output_Dir, exist_ok=True)

if Quality_Mode == "Discrete Qualities":
    parsed_q = [int(x) for x in re.findall(r"\d+", Discrete_Qualities)]
    qualities_list = list(dict.fromkeys([max(1, min(100, q)) for q in parsed_q]))
    q_range = None
    num_random = 0
    print(f"Generating with Discrete JPEG Qualities: {qualities_list}")
else:
    qualities_list = None
    q_range = (min(Quality_Min, Quality_Max), max(Quality_Min, Quality_Max))
    num_random = Random_Versions_Per_Crop
    print(f"Generating with Random Quality Range: [{q_range[0]}, {q_range[1]}] ({num_random} versions per crop)")

print("Scanning 'samples/' directory...")
total_generated = process_all_samples(
    samples_dir="samples",
    output_clean_dir=Clean_Output_Dir,
    output_noisy_dir=Noisy_Output_Dir,
    jpeg_qualities=qualities_list,
    quality_range=q_range,
    num_random_versions=num_random,
    num_crops=Crops_Per_Image,
    split_ratios=(0.8, 0.2, 0.0),
    seed=42,
)

print(f"\n✓ Sample Data Generation Complete!")
print(f"  Total Paired Samples Generated: {total_generated}")
print(f"  Clean Patches: {Clean_Output_Dir}")
print(f"  Noisy Patches: {Noisy_Output_Dir}")

In [ ]:
#@title 👁️ 4.2 Preview Generated Dataset Samples
#@markdown Displays random clean/noisy patch pairs from the generated dataset.

import random

clean_patches = glob.glob(os.path.join(Clean_Output_Dir, "*_clean.png"))
if not clean_patches:
    clean_patches = glob.glob("samples/processed/**/clean/*_clean.png", recursive=True)

if clean_patches:
    sample_clean_paths = random.sample(clean_patches, min(3, len(clean_patches)))
    fig, axes = plt.subplots(len(sample_clean_paths), 2, figsize=(10, 4 * len(sample_clean_paths)))
    if len(sample_clean_paths) == 1:
        axes = np.expand_dims(axes, 0)
        
    for idx, cp in enumerate(sample_clean_paths):
        np_path = cp.replace("_clean.png", "_noisy.png")
        if not os.path.exists(np_path):
            noisy_folder = os.path.dirname(cp).replace("clean", "noisy")
            np_path = os.path.join(noisy_folder, os.path.basename(cp).replace("_clean.png", "_noisy.png"))
            
        with Image.open(cp) as img_c:
            img_c_arr = np.array(img_c)
        with Image.open(np_path) as img_n:
            img_n_arr = np.array(img_n)
            
        axes[idx, 0].imshow(img_c_arr)
        axes[idx, 0].set_title(f"Clean Patch #{idx+1}: {os.path.basename(cp)[:25]}...")
        axes[idx, 0].axis("off")
        
        axes[idx, 1].imshow(img_n_arr)
        axes[idx, 1].set_title(f"Noisy Patch #{idx+1}: {os.path.basename(np_path)[:25]}...")
        axes[idx, 1].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("No generated patch pairs found. Run cell 4.1 first.")

---
## 🏋️ Section 5: Two-Stage Progressive Model Training Pipeline (Replaces GUI Tab 2)
Train the **RepAF-Denoise Net** using the robust two-stage training scheme:
1. **Stage 1 (Coarse & Frequency Training)**:
   - Progressive patch scaling from $128 \times 128 \to 256 \times 256$.
   - **Loss function**: `Stage1Loss` combines Charbonnier ($L_1$) loss with Sobel edge/gradient loss to prevent oversmoothing:
     $$\mathcal{L}_{\text{Stage 1}} = \sqrt{\|\hat{y} - y\|^2 + \epsilon^2} + \lambda_{\text{grad}} \|\nabla \hat{y} - \nabla y\|_1$$
2. **Stage 2 (Fine-Tuning)**:
   - Fixed large patches ($512 \times 512$) for high-frequency perceptual recovery.
   - **Loss function**: Direct `PSNRLoss` minimization:
     $$\mathcal{L}_{\text{PSNR}} = -10 \log_{10}\left(\frac{1}{\text{MSE}(\hat{y}, y) + \epsilon}\right)$$

In [ ]:
#@title 🏋️ 5.1 Configure & Launch Training Pipeline
#@markdown Set epochs and hyperparameters. For a quick dry-run test, check `Dry_Run`.

Stage_1_Epochs = 50 #@param {type:"integer"}
Stage_2_Epochs = 15 #@param {type:"integer"}
Batch_Size_Stage_1 = 32 #@param {type:"integer"}
Batch_Size_Stage_2 = 8 #@param {type:"integer"}
Learning_Rate = 2e-3 #@param {type:"number"}
Dry_Run = False #@param {type:"boolean"}

if Dry_Run:
    Stage_1_Epochs = 2
    Stage_2_Epochs = 1
    Batch_Size_Stage_1 = 4
    Batch_Size_Stage_2 = 2
    print("🏃 Dry run enabled: using minimal epochs for quick verification.")

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from dataset import DenoisingDataset, clear_image_cache
from losses import PSNRLoss, Stage1Loss
from metrics import calculate_psnr, calculate_ssim
from models import RepAFDenoiseNet
from train import get_device, get_progressive_patch_size

device = get_device()
print(f"\nInitializing Training on device: {device}")

# Datasets
train_dataset = DenoisingDataset(
    noisy_dir="samples/processed/train/noisy",
    clean_dir="samples/processed/train/clean",
    patch_size=128,
    is_train=True,
    num_synthetic_samples=64,
    cache=True,
)
val_dataset = DenoisingDataset(
    noisy_dir="samples/processed/val/noisy",
    clean_dir="samples/processed/val/clean",
    patch_size=256,
    is_train=False,
    num_synthetic_samples=32,
    cache=True,
)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, pin_memory=(device.type == "cuda"))

print(f"Train Dataset: {len(train_dataset)} samples | Val Dataset: {len(val_dataset)} samples")

model = RepAFDenoiseNet(c=40).to(device)
best_psnr = -float("inf")
save_dir = "pytorch models"
os.makedirs(save_dir, exist_ok=True)

# Training history trackers for visualization
history = {
    "stage1_loss": [], "stage1_val_psnr": [], "stage1_val_ssim": [],
    "stage2_loss": [], "stage2_val_psnr": [], "stage2_val_ssim": [],
}

# ----------------- STAGE 1 -----------------
print(f"\n{'='*20} Starting Stage 1 ({Stage_1_Epochs} epochs) {'='*20}")
optimizer_s1 = AdamW(model.parameters(), lr=Learning_Rate, betas=(0.9, 0.999), weight_decay=1e-4)
scheduler_s1 = CosineAnnealingLR(optimizer_s1, T_max=Stage_1_Epochs, eta_min=1e-6)
criterion_s1 = Stage1Loss(eps=1e-3, lambda_grad=0.05).to(device)
loader_s1 = DataLoader(train_dataset, batch_size=Batch_Size_Stage_1, shuffle=True, pin_memory=(device.type == "cuda"))

for epoch in range(1, Stage_1_Epochs + 1):
    t0 = time.time()
    patch_size = get_progressive_patch_size(epoch - 1, Stage_1_Epochs, min_size=128, max_size=256)
    train_dataset.set_patch_size(patch_size)
    
    model.train()
    epoch_loss = 0.0
    for noisy, clean in loader_s1:
        noisy, clean = noisy.to(device, non_blocking=True), clean.to(device, non_blocking=True)
        optimizer_s1.zero_grad(set_to_none=True)
        loss = criterion_s1(model(noisy), clean)
        loss.backward()
        optimizer_s1.step()
        epoch_loss += loss.item() * noisy.size(0)
        
    scheduler_s1.step()
    mean_loss = epoch_loss / max(1, len(train_dataset))
    
    # Validation
    model.eval()
    val_psnr, val_ssim, count = 0.0, 0.0, 0
    with torch.inference_mode():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device, non_blocking=True), clean.to(device, non_blocking=True)
            pred = torch.clamp(model(noisy), 0.0, 1.0)
            val_psnr += calculate_psnr(pred, clean) * noisy.size(0)
            val_ssim += calculate_ssim(pred, clean) * noisy.size(0)
            count += noisy.size(0)
    val_psnr /= max(1, count)
    val_ssim /= max(1, count)
    
    history["stage1_loss"].append(mean_loss)
    history["stage1_val_psnr"].append(val_psnr)
    history["stage1_val_ssim"].append(val_ssim)
    
    is_best = val_psnr > best_psnr
    if is_best:
        best_psnr = val_psnr
        torch.save(
            {"state_dict": model.state_dict(), "best_psnr": best_psnr, "stage": 1},
            os.path.join(save_dir, "best_model.pth"),
        )
        
    elapsed = time.time() - t0
    best_marker = " 🌟 BEST" if is_best else ""
    print(f"S1 [Epoch {epoch:03d}/{Stage_1_Epochs:03d}] Patch: {patch_size}x{patch_size} | "
          f"Loss: {mean_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | SSIM: {val_ssim:.4f}{best_marker} [{elapsed:.1f}s]")

del optimizer_s1, scheduler_s1, criterion_s1, loader_s1
if device.type == "cuda":
    torch.cuda.empty_cache()

# ----------------- STAGE 2 -----------------
print(f"\n{'='*20} Starting Stage 2 ({Stage_2_Epochs} epochs) {'='*20}")
train_dataset.set_patch_size(512)
loader_s2 = DataLoader(train_dataset, batch_size=Batch_Size_Stage_2, shuffle=True, pin_memory=(device.type == "cuda"))
optimizer_s2 = AdamW(model.parameters(), lr=Learning_Rate, betas=(0.9, 0.999), weight_decay=1e-4)
scheduler_s2 = CosineAnnealingLR(optimizer_s2, T_max=Stage_2_Epochs, eta_min=1e-6)
criterion_s2 = PSNRLoss().to(device)

for epoch in range(1, Stage_2_Epochs + 1):
    t0 = time.time()
    model.train()
    epoch_loss = 0.0
    for noisy, clean in loader_s2:
        noisy, clean = noisy.to(device, non_blocking=True), clean.to(device, non_blocking=True)
        optimizer_s2.zero_grad(set_to_none=True)
        loss = criterion_s2(model(noisy), clean)
        loss.backward()
        optimizer_s2.step()
        epoch_loss += loss.item() * noisy.size(0)
        
    scheduler_s2.step()
    mean_loss = epoch_loss / max(1, len(train_dataset))
    
    # Validation
    model.eval()
    val_psnr, val_ssim, count = 0.0, 0.0, 0
    with torch.inference_mode():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device, non_blocking=True), clean.to(device, non_blocking=True)
            pred = torch.clamp(model(noisy), 0.0, 1.0)
            val_psnr += calculate_psnr(pred, clean) * noisy.size(0)
            val_ssim += calculate_ssim(pred, clean) * noisy.size(0)
            count += noisy.size(0)
    val_psnr /= max(1, count)
    val_ssim /= max(1, count)
    
    history["stage2_loss"].append(mean_loss)
    history["stage2_val_psnr"].append(val_psnr)
    history["stage2_val_ssim"].append(val_ssim)
    
    is_best = val_psnr > best_psnr
    if is_best:
        best_psnr = val_psnr
        torch.save(
            {"state_dict": model.state_dict(), "best_psnr": best_psnr, "stage": 2},
            os.path.join(save_dir, "best_model.pth"),
        )
        
    elapsed = time.time() - t0
    best_marker = " 🌟 BEST" if is_best else ""
    print(f"S2 [Epoch {epoch:02d}/{Stage_2_Epochs:02d}] Patch: 512x512 | "
          f"PSNR Loss: {mean_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | SSIM: {val_ssim:.4f}{best_marker} [{elapsed:.1f}s]")

# Save final deployed model
clear_image_cache()
print(f"\n✓ Training Finished! Highest Validation PSNR Achieved: {best_psnr:.2f} dB")
print(f"  Best checkpoint stored at: pytorch models/best_model.pth")

In [ ]:
#@title 📊 5.2 Plot Training Loss & Validation Metrics
#@markdown Generates publication-ready training loss and validation PSNR/SSIM curves.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
axes[0].plot(range(1, len(history["stage1_loss"]) + 1), history["stage1_loss"], label="Stage 1 (Charbonnier + Sobel)", color="#1976d2", lw=2)
if history["stage2_loss"]:
    s2_epochs = range(len(history["stage1_loss"]) + 1, len(history["stage1_loss"]) + len(history["stage2_loss"]) + 1)
    axes[0].plot(s2_epochs, history["stage2_loss"], label="Stage 2 (PSNR Loss)", color="#e64a19", lw=2)
axes[0].set_title("Training Loss Progression", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# PSNR Plot
axes[1].plot(range(1, len(history["stage1_val_psnr"]) + 1), history["stage1_val_psnr"], label="Stage 1 Val PSNR", color="#388e3c", lw=2)
if history["stage2_val_psnr"]:
    s2_epochs = range(len(history["stage1_val_psnr"]) + 1, len(history["stage1_val_psnr"]) + len(history["stage2_val_psnr"]) + 1)
    axes[1].plot(s2_epochs, history["stage2_val_psnr"], label="Stage 2 Val PSNR", color="#fbc02d", lw=2)
axes[1].set_title(f"Validation PSNR Progression (Peak: {best_psnr:.2f} dB)", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

---
### 🏁 Conclusion & Next Steps
You have successfully:
- Tested interactive image denoising with artifact injection.
- Replaced the desktop GUI with a reproducible, scalable Google Colab environment.
- Fused multi-branch modules via structural re-parameterization.
- Exported the model to **FP32 ONNX**, **FP16 ONNX**, **QDQ INT8 ONNX**, and **PyTorch INT8**.
- Packaged all deployment artifacts for production.

For mobile or edge deployments, the exported ONNX models can directly run in C++, C#, Python, or TensorRT using the ONNX Runtime!